# Variational Quantum Simulation with analog QPUs

Here is a very basic example of Variational Quantum Simulation (VQS, as introduced in http://dx.doi.org/10.1038/s41586-019-1177-4) with `QutipQPU`.

Let us recall that Variational Quantum Simulation consists in finding the approximate ground state of a "target" Hamiltonian ``H_target`` using a variational state preparation parameterized by a time-dependent Hamiltonian (called `drive` below, also called a "resource" Hamiltonian in the literature).

## A basic example
In this example, the parameters we are going to optimize are the durations (parameterized by times $t_1$ and $t_2$) of the pulses of our resource Hamiltonian to prepare a trial state that minimizes the target energy (encoded in ``H_target``). 

The optimization of the parameters is carried out by a so-called 'variational plugin', ``ScipyMinimizePlugin`` (the same we also use for digital variational algorithms such as VQE or QAOA).

In this example, we use a basic noise model described in terms of Lindblad jump operators (``jump_operators`` below). 

Here is the full code of VQS:

In [ ]:
%matplotlib inline
from qat.core import Observable, Term, Schedule
from qat.core.variables import Variable, heaviside
from qat.qpus import QutipQPU
from qat.plugins import ScipyMinimizePlugin
from qat.hardware import HardwareModel

# Resource Hamiltonian
t = Variable("t", float)
t1 = Variable("t_1", float)
t2 = Variable("t_2", float)
tf = 3.0
drive = [(heaviside(t, 0., t1), Observable(2, pauli_terms=[Term(1, "X", [0]), Term(1, "X", [1])])), 
         (heaviside(t, t1, t2), Observable(2, pauli_terms=[Term(1, "ZZ", [0, 1])])),
         (heaviside(t, t2, tf), Observable(2, pauli_terms=[Term(1, "X", [0]), Term(1, "X", [1])]))] 
schedule = Schedule(drive=drive, tmax=tf)

# Target Hamiltonian
H_target = Observable(2, pauli_terms=[Term(1, 'XZ', [0, 1])])

# Parametric job
job = schedule.to_job(job_type="OBS", observable=H_target)

# Description of the stack
hardware_model = HardwareModel(jump_operators=[Observable(2, pauli_terms=[(Term(.1, 'Z', [0]))])])
qpu = QutipQPU(hardware_model=hardware_model)
optimizer_scipy = ScipyMinimizePlugin(method="COBYLA", x0=[0.4, 1.2])
stack = optimizer_scipy | qpu

# Simulation
res = stack.submit(job)
print("<H_target> =", res.value)

## Checking the exact result

We compare the variational energy we got with the target Hamiltonian's exact ground state energy.

In [ ]:
import numpy as np
from qat.fermion import SpinHamiltonian
H_target_mat = SpinHamiltonian(H_target.nbqbits, H_target.terms).get_matrix()
eigvals, eigvecs = np.linalg.eigh(H_target_mat)
print("E0 =", min(eigvals))

### Plot of the variational energy vs. the optimization step

We now plot the results, i.e. the convergence to the optimized energy.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 4), dpi=90)
plt.plot(eval(res.meta_data["optimization_trace"]), lw=3, label ="VQS energies")
plt.plot([min(eigvals) for _ in range(len(eval(res.meta_data["optimization_trace"])))],
         '--k', lw=3, label = "exact energy")
plt.grid()
plt.xlabel("VQS steps")
plt.ylabel("Energy")
plt.legend()

# Show the optimized parameters
params = eval(res.meta_data["parameters"])
print("Optimized parameters:\n" + str(params))

## Playing with the optimized schedule

In [ ]:
params_dic = {"t_1": params[0],   "t_2": params[1]}

# Let us check that the final energy is the expected one
job2= job(**params_dic)
res = qpu.submit(job2)
print("<H_target> =", res.value)

# let us check the probabilities of the final distribution
job3 = schedule.to_job(job_type="SAMPLE")(**params_dic)
res = qpu.submit(job3)
for sample in res:
    print(sample.state, sample.probability)

In [ ]:
sched = job2.schedule
sched.display()